# 06 · Baseline race model

Two parts, matching how the race simulator will work later:

1. **DNF model**: chance a driver doesn't take the chequered flag (retired, disqualified, didn't start). Lapped cars that finish count as finishers. Benchmark: the training-set DNF rate for everyone.
2. **Position model**: finishing order among the drivers who finished. Benchmark: finish in grid order.

Each is fitted as a **linear model** and a **gradient-boosted tree model**, trained on 2018–2024 and tested on 2025.

**Inputs**, all using only information available before the race:

| Input | Meaning |
|---|---|
| Grid, PitLaneStart | starting slot; a pit-lane start counts as last |
| QualiGapPct | this weekend's qualifying gap to teammate (%) |
| DriverForm, TeamForm | mean finishing rank in their last 5 finishes |
| RacePaceGapPct | mean race-pace gap to teammate over the last 5 dry races |
| DriverDNFRate, TeamDNFRate | share of DNFs in the last 10 races |
| CircuitPosChange | how much the order usually shuffles at this circuit (earlier editions) |
| CircuitDNFRate | DNF rate at this circuit in earlier editions |

**Limitations:** team form carries over between seasons even when the car changes a lot, and it restarts when a team rebrands (e.g. Racing Point → Aston Martin).

Run notebooks 00, 03 and 05 first.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Build the dataset
TEST_YEAR = 2025  # @param {type:"integer"}
WET_RULE = "tyres"  # @param ["tyres", "tyres_or_rain"]

import pandas as pd
import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest import load_quali
from f1sim.ingest_results import load_results
from f1sim.ingest_tracing import load_race_ti
from f1sim.features import teammate_gaps
from f1sim.race_pace import clean_dry_laps, teammate_race_gaps
from f1sim.baseline import (build_dataset, fit_evaluate, is_finished,
                            POSITION_FEATURES, DNF_FEATURES)

paths = get_paths()
results = load_results(paths)
quali_gaps = teammate_gaps(load_quali(paths))
race_gaps, _ = teammate_race_gaps(clean_dry_laps(load_race_ti("laps", paths), wet_rule=WET_RULE))
df = build_dataset(results, quali_gaps, race_gaps)

print(f"{len(df):,} driver-race rows | {df['RaceIdx'].nunique()} races | "
      f"{100 * df['Finished'].mean():.1f}% finished")
print("Missing values per input (%):")
(100 * df[sorted(set(POSITION_FEATURES + DNF_FEATURES))].isna().mean()).round(1)

In [ ]:
# @title Check the DNF rule: how each status was classified
status = results["Status"].value_counts().rename("Count").to_frame()
status["CountsAs"] = ["finished" if is_finished(s) else "DNF" for s in status.index]
status

In [ ]:
# @title Fit on earlier seasons, score on the test season
out = fit_evaluate(df, test_year=TEST_YEAR)
print(f"Train rows: {out['n_train_rows']:,} | test rows: {out['n_test_rows']:,}")
print("\nDNF model (lower LogLoss/Brier is better; AUC 0.5 = no better than chance):")
display(out["dnf_metrics"])
print("Position model, finishers only (higher Spearman, lower MAE is better):")
display(out["position_metrics"])
if len(out["position_unscored"]):
    print("Left out of the position scores (no grid position):")
    display(out["position_unscored"])

In [ ]:
# @title Race by race: how well each model ordered the finishers
per_race = out["position_per_race"].melt(id_vars=["Round", "Event"], var_name="Model",
                                         value_name="Spearman")
fig = px.line(per_race, x="Round", y="Spearman", color="Model", markers=True,
              hover_data=["Event"],
              title=f"{TEST_YEAR}: rank correlation between predicted and actual finishing order")
fig.show()

In [ ]:
# @title What drives the position predictions
coefs = out["position_linear_coefs"].rename("Coefficient").rename_axis("Input").reset_index()
px.bar(coefs, x="Coefficient", y="Input", orientation="h",
       title="Linear model: standardised coefficients (positive = finishes further back)"
       ).update_layout(yaxis={"categoryorder": "total ascending"}).show()

imp = out["position_gb_importance"].rename("Importance").rename_axis("Input").reset_index()
px.bar(imp, x="Importance", y="Input", orientation="h",
       title="Gradient boosting: permutation importance on the test season (positions of MAE lost)"
       ).update_layout(yaxis={"categoryorder": "total ascending"}).show()

In [ ]:
# @title One race: predicted vs actual finishing order
ROUND = 1  # @param {type:"integer"}

race = out["position_test_pred"]
race = race[race["Round"] == ROUND].sort_values("FinishRank")
print(race["Event"].iloc[0] if len(race) else f"No finishers found for round {ROUND}")
race.drop(columns=["Year", "Round", "Event"])

In [ ]:
# @title DNF model: highest predicted DNF chances in the test season
pred = out["dnf_test_pred"]
col = "P_DNF (Linear (logistic))"
pred.sort_values(col, ascending=False).head(20)[
    ["Round", "Event", "Abbreviation", "TeamName", "Grid", col, "DNF", "Status"]]